In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset, random_split, TensorDataset

import shap 

import numpy as np

import os

from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, roc_auc_score, accuracy_score, precision_score, recall_score, f1_score, roc_curve
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from tqdm import tqdm

import sys
sys.path.append('../src')

from preprocessing import *
from models import  *

from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
# Set global font sizes
plt.rcParams.update({
    'font.size': 18,              # Base font size
    'axes.titlesize': 20,         # Title font size
    'axes.labelsize': 15,         # Axes label font size
    'xtick.labelsize': 15,        # X-tick label font size
    'ytick.labelsize': 15,        # Y-tick label font size
    'legend.fontsize': 16,        # Legend font size
    'figure.titlesize': 18,       # Figure title size
    'figure.dpi': 300,            # Figure DPI for high resolution
    'savefig.dpi': 300            # Save figure DPI
})

# Make lines thicker
plt.rcParams['lines.linewidth'] = 2.5
plt.rcParams['axes.linewidth'] = 1.5
plt.rcParams['xtick.major.width'] = 1.5
plt.rcParams['ytick.major.width'] = 1.5

# Increase marker size default
plt.rcParams['lines.markersize'] = 8

In [ ]:
dfs = get_dfs(os.path.dirname(os.getcwd()))
static_df = create_static_df(dfs)
medication_df = create_medication_df(dfs)
vitals_ca, vitals_lab = create_vitals_df(dfs)

ts_data = create_ts_data(vitals_ca, vitals_lab, medication_df, merge_lab=True, merge_med=True, static_df=static_df)
#forward_fill_imputation(ts_data) 

#notes = create_notes_df(dfs, filename='../data/embeddings/emb_gte.npy')
notes = create_notes_df(dfs, filename='../data/embeddings/emb_med_gte_simcse_en_ger.npy')
#notes = create_notes_df(dfs, filename=None)

full_dataset = NephroCAGEDataset(static_df=static_df, ts_data=ts_data, notes_df=notes, biopsy_df=dfs['biopsy'])
datapoints_limit = len(full_dataset)
#datapoints_limit = 320
dataset = Subset(full_dataset, indices=list(range(datapoints_limit)))
ts_scaler = full_dataset.ts_scaler
static_scaler = full_dataset.scaler

In [ ]:
# Split dataset into training and test sets (80% train, 20% test)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

batch_size = 16
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

vanilla_lstm = VanillaTimeSeriesEncoder()
att_encoder = TimeAwareAttentionEncoder(use_temporal_attention=True)
model = MultiModal(vanilla_lstm, categorical_cardinalities=full_dataset.categorical_cardinalities, use_static=True, use_notes=False).to(device)
predict_steps_ahead = 1

model.load_state_dict(torch.load('../models/vanilla.pt', weights_only=True))

In [ ]:
def extract_horizon_reprs(
    dataloader, 
    model, 
    horizons, 
    label_key, 
    rel_days_key=None,
    min_history_days=90,
    max_days=180,
    max_samples_per_patient=100
):
    """
    - Feeds each patient's entire time series (minus last step) in one pass.
    - Extracts hidden representations for each time step from model output.
    - For each horizon H, determines if the event occurs within H days from that step.
    - Only includes time steps between min_history_days and max_days.
    - Takes up to max_samples_per_patient samples per patient (first N valid samples).

    If `label_key` corresponds to a multi-day event list (like "rej_rel_days"), then `rel_days_key`
    can be left None (ignored), and we will handle the logic differently.
    
    Parameters:
    -----------
    dataloader : DataLoader
        The PyTorch dataloader yielding patient data batches
    model : torch.nn.Module
        The trained model to extract representations from
    horizons : list
        List of horizon values (in days) to consider
    label_key : str
        Key in batch dictionary for labels
    rel_days_key : str, optional
        Key for relative days to event (only for single-event labels)
    min_history_days : int, default=90
        Minimum number of days of history required
    max_days : int, default=180
        Maximum number of days to include in the dataset
    max_samples_per_patient : int, default=5
        Maximum number of samples to include per patient
    """
    from tqdm import tqdm
    
    model.eval()

    # For each horizon, prepare storage for hidden reps, labels, day-of-step, patient_id
    hr_repr = {H: [] for H in horizons}
    hr_label = {H: [] for H in horizons}
    hr_days  = {H: [] for H in horizons}
    hr_pids  = {H: [] for H in horizons}
    
    # Keep track of samples per patient for each horizon
    patient_sample_counts = {H: {} for H in horizons}

    all_pids = set()
    positive_pids = set()

    with torch.no_grad():
        # Add progress bar for the dataloader iteration
        for batch in tqdm(dataloader, desc="Processing patients"):
            pid  = batch['patient_id']
            slen = batch['seq_len']

            # Static features
            cat_static = batch['static_categorical_features'].to(device)
            num_static = batch['static_numerical_features'].to(device)

            # Time series
            full_ts    = batch['ts_features'].to(device)  # shape (B, T, F)
            timesteps  = batch['timesteps'].to(device)    # shape (B, T)
            mask_      = batch['mask'].to(device)         # shape (B, T)

            # --- Convert single-element Tensors in label_key to float/list ---
            raw_labels_data = batch[label_key]  # shape (B,)
            labels_data = []
            for val in raw_labels_data:
                if isinstance(val, torch.Tensor):
                    # If it's a single element, convert to float
                    if val.numel() == 1:
                        val = float(val.item())
                    else:
                        # If multi-element, convert to NumPy or list
                        val = val.cpu().numpy()
                # Otherwise, val can be float, int, list, etc.
                labels_data.append(val)

            # If single-event usage, we also have rel_days_key => shape (B,)
            if rel_days_key and rel_days_key in batch:
                raw_rel_days_data = batch[rel_days_key]
                rel_days_data = []
                for dval in raw_rel_days_data:
                    if isinstance(dval, torch.Tensor):
                        if dval.numel() == 1:
                            dval = float(dval.item())
                        else:
                            dval = dval.cpu().numpy()
                    rel_days_data.append(dval)
                rel_days_data = np.array(rel_days_data)
            else:
                rel_days_data = None  # We'll handle multi-day logic below

            # If using notes
            notes_embeddings = batch['notes_embeddings'].to(device)
            notes_timesteps  = batch['notes_timesteps'].to(device)
            notes_mask       = batch['notes_mask'].to(device)

            B, T, F = full_ts.shape

            # Loop over each patient in this batch
            for i in range(B):
                patient_id_i = pid[i]
                all_pids.add(patient_id_i)

                # label_or_list => single-event (float 0/1) or multi-event list
                label_or_list = labels_data[i]

                # Keep track of which patients have at least one event
                if isinstance(label_or_list, (int, float, np.number)):
                    # single label
                    if label_or_list == 1:
                        positive_pids.add(patient_id_i)
                elif isinstance(label_or_list, (list, np.ndarray)):
                    # multiple days => if not empty => event
                    if len(label_or_list) > 0:
                        positive_pids.add(patient_id_i)
                elif label_or_list is not None:
                    # Unknown type
                    raise TypeError(f"Unsupported label type: {type(label_or_list)}")

                # If the sequence is too short
                if slen[i] < 2:
                    continue
                
                # Check if we already have max samples for this patient for all horizons
                if max_samples_per_patient > 0:
                    all_horizons_at_max = True
                    for H in horizons:
                        if patient_id_i not in patient_sample_counts[H] or patient_sample_counts[H][patient_id_i] < max_samples_per_patient:
                            all_horizons_at_max = False
                            break
                    
                    if all_horizons_at_max:
                        continue  # Skip this patient altogether if already at max for all horizons

                # Slice the valid portion of the time series: [0..slen[i]-1]
                seq_len_i = slen[i].item()
                ts_i = full_ts[i:i+1, :seq_len_i, :]   # (1, seq_len_i, F)
                tm_i = timesteps[i:i+1, :seq_len_i]    # (1, seq_len_i)
                mk_i = mask_[i:i+1, :seq_len_i]        # (1, seq_len_i)

                # Model input => omit last step from time series
                inp_seq = ts_i[:, :-1, :]              # (1, seq_len_i-1, F)
                elapsed_times = tm_i[:, 1:] - tm_i[:, :-1]
                inp_mask = mk_i[:, :-1]

                notes_emb_i = notes_embeddings[i:i+1]
                notes_ts_i  = notes_timesteps[i:i+1]
                notes_mk_i  = notes_mask[i:i+1]

                # Forward pass
                out, lstm_out, _, _ = model(
                    x=inp_seq,
                    elapsed_times=elapsed_times,     # (1, seq_len_i - 1)
                    timesteps=tm_i[:, :-1],          # (1, seq_len_i - 1)
                    notes_embeddings=notes_emb_i,
                    notes_timesteps=notes_ts_i,
                    static_features=(cat_static[i:i+1], num_static[i:i+1]),
                    mask=inp_mask,                   # (1, seq_len_i - 1)
                    notes_mask=notes_mk_i
                )
                # lstm_out => shape (1, seq_len_i-1, hidden_size)

                # Time array for all steps
                time_arr = tm_i.cpu().numpy().flatten()  # (seq_len_i,)
                
                # Initialize counts for this patient if not present
                for H in horizons:
                    if patient_id_i not in patient_sample_counts[H]:
                        patient_sample_counts[H][patient_id_i] = 0
                
                # Loop through time steps and only process until we reach max_samples for each horizon
                for k in range(seq_len_i - 1):
                    cur_day = time_arr[k+1]  # day of the (k+1)-th step
                    
                    # Skip if outside the desired range
                    if cur_day < min_history_days or cur_day > max_days:
                        continue
                    
                    # Get representation for this time step
                    rep_ = lstm_out[0, k, :].cpu().numpy()
                    
                    # Check for each horizon if we still need more samples
                    any_horizon_needs_samples = False
                    for H in horizons:
                        if patient_sample_counts[H][patient_id_i] < max_samples_per_patient:
                            any_horizon_needs_samples = True
                            break
                    
                    if not any_horizon_needs_samples:
                        break  # Exit time step loop if all horizons have enough samples
                    
                    # Process for each horizon that still needs samples
                    for H in horizons:
                        # Skip if already have max samples for this horizon
                        if patient_sample_counts[H][patient_id_i] >= max_samples_per_patient:
                            continue
                        
                        # If single-event logic is in play
                        if rel_days_data is not None:
                            event_label = label_or_list     # 0 or 1
                            event_day   = rel_days_data[i]  # single day
                            if (event_label == 1) and (0 < (event_day - cur_day) <= H):
                                label_ = 1
                            else:
                                label_ = 0

                        # Else multi-event logic (like rejections)
                        else:
                            # label_or_list is a list of event days or None
                            if isinstance(label_or_list, (list, np.ndarray)) and len(label_or_list) > 0:
                                # label_ = 1 if any day d in label_or_list satisfies (0 < d - cur_day <= H)
                                label_ = int(any(0 < (d - cur_day) <= H for d in label_or_list))
                            else:
                                label_ = 0

                        # Store
                        hr_repr[H].append(rep_)
                        hr_label[H].append(label_)
                        hr_days[H].append(cur_day)
                        hr_pids[H].append(patient_id_i)
                        
                        # Update sample count
                        patient_sample_counts[H][patient_id_i] += 1

    # Convert lists to numpy arrays for convenience
    for H in horizons:
        hr_repr[H] = np.array(hr_repr[H])
        hr_label[H] = np.array(hr_label[H])
        hr_days[H]  = np.array(hr_days[H])
        hr_pids[H]  = np.array(hr_pids[H])

    # Calculate and print statistics
    for H in horizons:
        total_patients = len(patient_sample_counts[H])
        avg_samples = np.mean([patient_sample_counts[H][pid] for pid in patient_sample_counts[H]])
        max_samples = max([patient_sample_counts[H][pid] for pid in patient_sample_counts[H]]) if patient_sample_counts[H] else 0
        
        print(f"Horizon {H}: {total_patients} patients, avg {avg_samples:.1f} samples/patient, max {max_samples} samples/patient")

    print(f"Total unique patients: {len(all_pids)}, patients with event: {len(positive_pids)}")
    return hr_repr, hr_label, hr_days, hr_pids

In [ ]:
def undersample_and_split(X, y, test_size=0.2, undersample_ratio=0.2, random_state=42):
    """
    Undersample the majority class and perform stratified train-test split.
    
    Parameters:
    -----------
    X : numpy.ndarray
        Feature matrix
    y : numpy.ndarray
        Labels array
    test_size : float, default=0.2
        Proportion of the dataset to include in the test split
    undersample_ratio : float, default=0.2
        Fraction of majority class samples to keep
    random_state : int, default=42
        Random seed for reproducibility
    
    Returns:
    --------
    X_train, X_test, y_train, y_test : arrays
        Undersampled and split data
    """
    # Get the indices of negative and positive samples
    neg_indices = np.where(y == 0)[0]
    pos_indices = np.where(y == 1)[0]
    
    # Randomly select a subset of negative samples
    np.random.seed(random_state)
    neg_indices_subset = np.random.choice(neg_indices, size=int(undersample_ratio * len(neg_indices)), replace=False)
    
    # Combine with all positive samples
    selected_indices = np.concatenate([neg_indices_subset, pos_indices])
    
    # Create the balanced subset
    X_subset = X[selected_indices]
    y_subset = y[selected_indices]
    
    # Perform stratified train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X_subset, y_subset, 
        test_size=test_size, 
        stratify=y_subset,
        random_state=random_state
    )
    
    return X_train, X_test, y_train, y_test

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

# Modified cell for extracting and processing representations
horizons = [90]
min_history_days = 10
max_days = 20000
undersample_ratio = 0.2
test_size = 0.2
random_state = 42

print("Embedding patients...")
# Extract representations using the original function
train_graft_repr, train_graft_lbl, train_graft_days, train_graft_pids = extract_horizon_reprs(train_dataloader, model, horizons, "graft_loss_label", "loss_rel_days", min_history_days, max_days)
test_graft_repr, test_graft_lbl, test_graft_days, test_graft_pids = extract_horizon_reprs(test_dataloader, model, horizons, "graft_loss_label", "loss_rel_days", min_history_days, max_days)
train_rej_repr, train_rej_lbl, train_rej_days, train_rej_pids = extract_horizon_reprs(train_dataloader, model, horizons, "rej_rel_days", None, min_history_days, max_days)
test_rej_repr, test_rej_lbl, test_rej_days, test_rej_pids = extract_horizon_reprs(test_dataloader, model, horizons, "rej_rel_days", None, min_history_days, max_days)
train_mort_repr, train_mort_lbl, train_mort_days, train_mort_pids = extract_horizon_reprs(train_dataloader, model, horizons, "death_label", "death_rel_days", min_history_days, max_days)
test_mort_repr, test_mort_lbl, test_mort_days, test_mort_pids = extract_horizon_reprs(test_dataloader, model, horizons, "death_label", "death_rel_days", min_history_days, max_days)

# Initialize dictionaries with same structure but with _p suffix
train_graft_repr_p = {H: None for H in horizons}
test_graft_repr_p = {H: None for H in horizons}
train_graft_lbl_p = {H: None for H in horizons}
test_graft_lbl_p = {H: None for H in horizons}

train_rej_repr_p = {H: None for H in horizons}
test_rej_repr_p = {H: None for H in horizons}
train_rej_lbl_p = {H: None for H in horizons}
test_rej_lbl_p = {H: None for H in horizons}

train_mort_repr_p = {H: None for H in horizons}
test_mort_repr_p = {H: None for H in horizons}
train_mort_lbl_p = {H: None for H in horizons}
test_mort_lbl_p = {H: None for H in horizons}

# Print stats and process each dataset for all horizons
for H in horizons:
    print(f"\n===== Original Extracted Features for Horizon {H} days =====")
    print(f"Graft Loss Training: {train_graft_repr[H].shape}, Positive: {sum(train_graft_lbl[H])}, Ratio: {sum(train_graft_lbl[H])/len(train_graft_lbl[H]):.4f}")
    print(f"Graft Loss Testing: {test_graft_repr[H].shape}, Positive: {sum(test_graft_lbl[H])}, Ratio: {sum(test_graft_lbl[H])/len(test_graft_lbl[H]):.4f}")
    print(f"Rejection Training: {train_rej_repr[H].shape}, Positive: {sum(train_rej_lbl[H])}, Ratio: {sum(train_rej_lbl[H])/len(train_rej_lbl[H]):.4f}")
    print(f"Rejection Testing: {test_rej_repr[H].shape}, Positive: {sum(test_rej_lbl[H])}, Ratio: {sum(test_rej_lbl[H])/len(test_rej_lbl[H]):.4f}")
    print(f"Mortality Training: {train_mort_repr[H].shape}, Positive: {sum(train_mort_lbl[H])}, Ratio: {sum(train_mort_lbl[H])/len(train_mort_lbl[H]):.4f}")
    print(f"Mortality Testing: {test_mort_repr[H].shape}, Positive: {sum(test_mort_lbl[H])}, Ratio: {sum(test_mort_lbl[H])/len(test_mort_lbl[H]):.4f}")
    
    # Process graft loss data
    print(f"\n--- Processing Graft Loss data for horizon {H} ---")
    # Combine train and test to get a full dataset before new stratified split
    X_graft = np.vstack((train_graft_repr[H], test_graft_repr[H]))
    y_graft = np.concatenate((train_graft_lbl[H], test_graft_lbl[H]))
    
    # Get the indices of negative and positive samples
    neg_indices = np.where(y_graft == 0)[0]
    pos_indices = np.where(y_graft == 1)[0]
    
    # Randomly select subset of negative samples
    np.random.seed(random_state)
    neg_indices_subset = np.random.choice(neg_indices, size=int(undersample_ratio * len(neg_indices)), replace=False)
    
    # Combine with all positive samples
    selected_indices = np.concatenate([neg_indices_subset, pos_indices])
    
    # Create the balanced subset
    X_subset = X_graft[selected_indices]
    y_subset = y_graft[selected_indices]
    
    X_train, X_test, y_train, y_test = train_test_split(
        X_subset, y_subset, 
        test_size=test_size, 
        stratify=y_subset,
        random_state=random_state
    )
    
    # Store processed data with _p suffix
    train_graft_repr_p[H] = X_train
    test_graft_repr_p[H] = X_test
    train_graft_lbl_p[H] = y_train
    test_graft_lbl_p[H] = y_test
    
    print(f"Undersampled Graft Loss:")
    print(f"Train: {len(train_graft_lbl_p[H])} samples, {sum(train_graft_lbl_p[H])} positives ({sum(train_graft_lbl_p[H])/len(train_graft_lbl_p[H]):.4f})")
    print(f"Test: {len(test_graft_lbl_p[H])} samples, {sum(test_graft_lbl_p[H])} positives ({sum(test_graft_lbl_p[H])/len(test_graft_lbl_p[H]):.4f})")
    
    # Process rejection data
    print(f"\n--- Processing Rejection data for horizon {H} ---")
    X_rej = np.vstack((train_rej_repr[H], test_rej_repr[H]))
    y_rej = np.concatenate((train_rej_lbl[H], test_rej_lbl[H]))
    
    # Get the indices of negative and positive samples
    neg_indices = np.where(y_rej == 0)[0]
    pos_indices = np.where(y_rej == 1)[0]
    
    # Randomly select subset of negative samples
    np.random.seed(random_state)
    neg_indices_subset = np.random.choice(neg_indices, size=int(undersample_ratio * len(neg_indices)), replace=False)
    
    # Combine with all positive samples
    selected_indices = np.concatenate([neg_indices_subset, pos_indices])
    
    # Create the balanced subset
    X_subset = X_rej[selected_indices]
    y_subset = y_rej[selected_indices]
    
    # Perform stratified train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X_subset, y_subset, 
        test_size=test_size, 
        stratify=y_subset,
        random_state=random_state
    )
    
    # Store processed data with _p suffix
    train_rej_repr_p[H] = X_train
    test_rej_repr_p[H] = X_test
    train_rej_lbl_p[H] = y_train
    test_rej_lbl_p[H] = y_test
    
    print(f"Undersampled Rejection:")
    print(f"Train: {len(train_rej_lbl_p[H])} samples, {sum(train_rej_lbl_p[H])} positives ({sum(train_rej_lbl_p[H])/len(train_rej_lbl_p[H]):.4f})")
    print(f"Test: {len(test_rej_lbl_p[H])} samples, {sum(test_rej_lbl_p[H])} positives ({sum(test_rej_lbl_p[H])/len(test_rej_lbl_p[H]):.4f})")
    
    # Process mortality data
    print(f"\n--- Processing Mortality data for horizon {H} ---")
    X_mort = np.vstack((train_mort_repr[H], test_mort_repr[H]))
    y_mort = np.concatenate((train_mort_lbl[H], test_mort_lbl[H]))
    
    # Get the indices of negative and positive samples
    neg_indices = np.where(y_mort == 0)[0]
    pos_indices = np.where(y_mort == 1)[0]
    
    # Randomly select subset of negative samples
    np.random.seed(random_state)
    neg_indices_subset = np.random.choice(neg_indices, size=int(undersample_ratio * len(neg_indices)), replace=False)
    
    # Combine with all positive samples
    selected_indices = np.concatenate([neg_indices_subset, pos_indices])
    
    # Create the balanced subset
    X_subset = X_mort[selected_indices]
    y_subset = y_mort[selected_indices]
    
    # Perform stratified train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X_subset, y_subset, 
        test_size=test_size, 
        stratify=y_subset,
        random_state=random_state
    )
    
    # Store processed data with _p suffix
    train_mort_repr_p[H] = X_train
    test_mort_repr_p[H] = X_test
    train_mort_lbl_p[H] = y_train
    test_mort_lbl_p[H] = y_test
    
    print(f"Undersampled Mortality:")
    print(f"Train: {len(train_mort_lbl_p[H])} samples, {sum(train_mort_lbl_p[H])} positives ({sum(train_mort_lbl_p[H])/len(train_mort_lbl_p[H]):.4f})")
    print(f"Test: {len(test_mort_lbl_p[H])} samples, {sum(test_mort_lbl_p[H])} positives ({sum(test_mort_lbl_p[H])/len(test_mort_lbl_p[H]):.4f})")

Horizon 90: 677 patients, avg 75.5 samples/patient, max 100 samples/patient
Total unique patients: 677, patients with event: 209

===== Original Extracted Features for Horizon 90 days =====
Graft Loss Training: (205302, 512), Positive: 3181, Ratio: 0.0155
Graft Loss Testing: (51104, 512), Positive: 686, Ratio: 0.0134
Rejection Training: (205302, 512), Positive: 2659, Ratio: 0.0130
Rejection Testing: (51104, 512), Positive: 797, Ratio: 0.0156
Mortality Training: (205302, 512), Positive: 4266, Ratio: 0.0208
Mortality Testing: (51104, 512), Positive: 1051, Ratio: 0.0206

--- Processing Graft Loss data for horizon 90 ---
Undersampled Graft Loss:
Train: 43499 samples, 3094 positives (0.0711)
Test: 10875 samples, 773 positives (0.0711)

--- Processing Rejection data for horizon 90 ---
Undersampled Rejection:
Train: 43236 samples, 2765 positives (0.0640)
Test: 10810 samples, 691 positives (0.0639)

--- Processing Mortality data for horizon 90 ---
Undersampled Mortality:
Train: 44427 samples, 

In [ ]:
def train_and_eval_logistic(X_train, y_train, X_test, y_test, event_name="Event"):
    # Create plots directory if it doesn't exist
    os.makedirs("../plots", exist_ok=True)
    
    # Train model
    clf = LogisticRegression(class_weight={0:1, 1:1}, max_iter=1000).fit(X_train, y_train)
    y_pred_proba = clf.predict_proba(X_test)[:, 1]
   
    threshold = .5
    y_pred = (y_pred_proba >= threshold).astype(int)
    
    # Calculate metrics
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    auc = roc_auc_score(y_test, y_pred_proba)
    
    print(f"\n{event_name} Prediction:")
    print(f"AUC = {auc:.4f}")
    print(f"Acc = {accuracy_score(y_test, y_pred):.4f}")
    print(f"Prec = {precision_score(y_test, y_pred):.4f}")
    print(f"Recall (Sensitivity) = {recall_score(y_test, y_pred):.4f}")
    print(f"Specificity = {specificity:.4f}")
    print(f"F1 = {f1_score(y_test, y_pred):.4f}")
    
    # Plot and save ROC curve
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='blue', lw=2)
    plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    #plt.legend(loc="lower right")
    
    # Save the figure
    filename = f"../plots/{event_name.replace('@', '_')}_roc.png"
    #plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"ROC curve saved to {filename}")
    
    return clf, auc

# Run for all horizons and events
for H in horizons:
    train_and_eval_logistic(train_graft_repr_p[H], train_graft_lbl_p[H], test_graft_repr_p[H], test_graft_lbl_p[H], event_name=f"GraftLoss@{H}")
    #train_and_eval_logistic(train_mort_repr[H], train_mort_lbl[H], test_mort_repr[H], test_mort_lbl[H], event_name=f"Mortality@{H}")
    #train_and_eval_logistic(train_rej_repr[H], train_rej_lbl[H], test_rej_repr[H], test_rej_lbl[H], event_name=f"Rejection@{H}")

In [ ]:
def plot_calibration_comparison(y_true, y_pred_proba, n_bins=10, 
                               model_name="Logistic Regression", method="both"):

    # Create a simple figure
    plt.figure(figsize=(10, 8))
    
    prob_true_orig, prob_pred_orig = calibration_curve(y_true, y_pred_proba, n_bins=n_bins)
    
    bin_edges = np.linspace(0, 1, n_bins + 1)
    bin_indices_orig = np.digitize(y_pred_proba, bin_edges[:-1])
    bin_counts_orig = np.bincount(bin_indices_orig, minlength=n_bins+1)[1:]
    
    brier_orig = brier_score_loss(y_true, y_pred_proba)
    
    y_pred_proba_platt = None
    prob_true_platt = None
    prob_pred_platt = None
    bin_counts_platt = None
    brier_platt = None
    
    y_pred_proba_isotonic = None
    prob_true_isotonic = None
    prob_pred_isotonic = None
    bin_counts_isotonic = None
    brier_isotonic = None
        
    reshaped_probs = y_pred_proba.reshape(-1, 1)
    
    # Platt scaling
    if method in ["platt", "both"]:
        platt_scaler = LogisticRegression(C=1.0, solver='lbfgs')
        platt_scaler.fit(reshaped_probs, y_true)
        y_pred_proba_platt = platt_scaler.predict_proba(reshaped_probs)[:, 1]
        
        # Calculate calibration curve for Platt scaled model
        prob_true_platt, prob_pred_platt = calibration_curve(y_true, y_pred_proba_platt, n_bins=n_bins)
        
        # Calculate bin counts for Platt scaled model
        bin_indices_platt = np.digitize(y_pred_proba_platt, bin_edges[:-1])
        bin_counts_platt = np.bincount(bin_indices_platt, minlength=n_bins+1)[1:]
        
        # Calculate Platt Brier score
        brier_platt = brier_score_loss(y_true, y_pred_proba_platt)
    
    # Isotonic Regression
    if method in ["isotonic", "both"]:
        isotonic_scaler = IsotonicRegression(out_of_bounds='clip')
        isotonic_scaler.fit(y_pred_proba, y_true)
        y_pred_proba_isotonic = isotonic_scaler.predict(y_pred_proba)
        
        # Calculate calibration curve for Isotonic model
        prob_true_isotonic, prob_pred_isotonic = calibration_curve(y_true, y_pred_proba_isotonic, n_bins=n_bins)
        
        # Calculate bin counts for Isotonic model
        bin_indices_isotonic = np.digitize(y_pred_proba_isotonic, bin_edges[:-1])
        bin_counts_isotonic = np.bincount(bin_indices_isotonic, minlength=n_bins+1)[1:]
        
        # Calculate Isotonic Brier score
        brier_isotonic = brier_score_loss(y_true, y_pred_proba_isotonic)
    
    # Plot perfectly calibrated line
    plt.plot([0, 1], [0, 1], 'k--', label='Perfectly Calibrated')
    
    # Plot original calibration curve
    plt.scatter(prob_pred_orig, prob_true_orig, s=100, alpha=0.7, color='blue',
               label=f'Original (Brier score: {brier_orig:.4f})')
    plt.plot(prob_pred_orig, prob_true_orig, '-o', linewidth=2, color='blue')
    
    # Plot Platt scaled calibration curve if available
    if method in ["platt", "both"] and prob_pred_platt is not None:
        plt.scatter(prob_pred_platt, prob_true_platt, s=100, alpha=0.7, color='red',
                   label=f'Platt Scaled (Brier score: {brier_platt:.4f})')
        plt.plot(prob_pred_platt, prob_true_platt, '-o', linewidth=2, color='red')
    
    # Plot Isotonic calibration curve if available
    if method in ["isotonic", "both"] and prob_pred_isotonic is not None:
        plt.scatter(prob_pred_isotonic, prob_true_isotonic, s=100, alpha=0.7, color='green',
                   label=f'Isotonic (Brier score: {brier_isotonic:.4f})')
        plt.plot(prob_pred_isotonic, prob_true_isotonic, '-o', linewidth=2, color='green')
    
    # Add plot details
    plt.title(f'Calibration Plot - {n_bins} bins', fontsize=14)
    plt.xlabel('Predicted Probability', fontsize=12)
    plt.ylabel('True Probability (Fraction of Positives)', fontsize=12)
    plt.xlim([0, 1])
    plt.ylim([0, 1])
    plt.grid(True, alpha=0.3)
    plt.legend(loc='upper left')
    
    plt.tight_layout()
    return plt.gcf()

In [ ]:
H = 90  # horizon
clf = LogisticRegression(class_weight={0:1, 1:4}, max_iter=1000).fit(train_graft_repr_p[H], train_graft_lbl_p[H])
y_pred_proba = clf.predict_proba(test_graft_repr_p[H])[:, 1]

fig = plot_calibration_comparison(
    #train_graft_lbl[H], y_pred_proba,
    test_graft_lbl_p[H], y_pred_proba,
    n_bins=20,
    method="platt"
)

plt.savefig("../plots/calibration_gl90.jpg")
plt.show()

In [ ]:
H = 90  # horizon
clf = LogisticRegression(class_weight={0:1, 1:5}, max_iter=1000).fit(train_rej_repr_p[H], train_rej_lbl_p[H])
y_pred_proba = clf.predict_proba(test_rej_repr_p[H])[:, 1]

fig = plot_calibration_comparison(
    #train_graft_lbl[H], y_pred_proba,
    test_rej_lbl_p[H], y_pred_proba,
    n_bins=20,
    method="platt"
)

plt.savefig("../plots/calibration_gr90")
plt.show()